# Recursive Descent Parsing

Recursive descent parsing, as a technique, emerged in the late 1950s to early 1960s alongside the development of formal language theory and early compiler design.

It was not introduced as a named method in a single, widely cited publication, but rather evolved organically as a practical approach to parsing context-free grammars—particularly LL(1) grammars.

Niklaus Wirth's work in the 1970s, The Pascal Report (used to define the Pascal programming language) his textbook, Algorithms + Data Structures = Programs, helped popularize the method as both a pedagogical and practical approach.

While parsing more advanced languages is best left to parsing tools and compiler toolkits, knowing this technique helps you to understand how compiilers and interpreters read your source code and produce results.

## How it Works

In this example, we develop a recursive descent parser that evaluates arithmetic expressions. The expressions include numeric literals (integers and floating-point numbers), unary operators `+` and `-`, binary operators `+`, `-`, `*`, `/`, and `**`, and parentheses `(` and `)` for grouping.

Each grammar rule becomes a recursive function. These functions process tokens from the input and call each other according to the structure of the grammar. The design ensures correct operator precedence and associativity, enabling evaluation of complex expressions like `3 + 4 * 2 / (1 - 5) ** 2 ** 3`




## Problem Statement

The goal is to implement a parser that converts a string representing a mathematical expression into its numeric value.

The language supports integers, floating-point numbers, parentheses, and the full suite of arithmetic operations.

Unary operators must bind tightly.

Exponentiation is right-associative, and parentheses override precedence.

The parser should also provide useful error messages when input is malformed.


## Understanding the Recursion

The grammar of the language is defined recursively with the following rules:

```
expr        ::= term (('+' | '-') term)*
term        ::= factor (('*' | '/') factor)*
factor      ::= unary ('**' factor)*
unary       ::= ('+' | '-') unary | primary
primary     ::= NUMBER | '(' expr ')'
```

This grammar ensures correct precedence and associativity:


- `expr` handles addition and subtraction
- `term` handles multiplication and division
- `factor` handles exponentiation (right-associative)
- `unary` handles unary `+` and `-`
- `primary` handles numeric values and parentheses

Rules higher in the grammar have the *lowest* precedence.

Rules lower in the grammar have the *highest* precedence.

Note that grammar rules can be written in any order so it is always a good practice to arrange your grammar rules in proper order to aid in understanding the hierarchy of rules within a particular category.

In the case of our language, it only has arithmetic expressions.


## Tokenization using Regular Expressions

To begin parsing, the input must be split into tokens. Without tokens, we would need to process input one character at a time, which is not effective. A token is a meaningful element such as a number, operator, or parenthesis. We use regular expressions to define token patterns.

The tokenizer is responsible for converting a raw input string into a structured sequence of tokens.

Each token represents a meaningful unit in the language, such as a number, an operator, or a parenthesis.

Tokenization is the first step in parsing and allows the recursive parser to operate on a predictable input format instead of directly navigating the original text.
In our implementation, we define a list of token specifications using regular expressions.

These patterns match numeric literals, the five operator symbols, and grouping parentheses.
Whitespace is skipped entirely.

The tokenizer scans the input string using Python’s `re.finditer` function and constructs a list of `Token` objects.
One subtle design choice in this tokenizer is the handling of exponentiation.

Because `**` is not a single character, we scan for consecutive `*` tokens and then merge them into a single `**` token.
This approach keeps the tokenizer simple while still allowing the parser to handle `**` as a distinct operator later in the grammar.
The resulting token list is passed to the parser, which uses it to construct and evaluate the expression in a structured way.


In [1]:
import re

class Token:
    def __init__(self, type_, value):
        self.type = type_
        self.value = value
    def __repr__(self):
        return f"Token({self.type}, {self.value})"

def tokenize(expression):
    token_spec = [
        ('NUMBER',  r'\d+(\.\d+)?'),
        ('PLUS',    r'\+'),
        ('MINUS',   r'-'),
        ('STAR',    r'\*'),
        ('SLASH',   r'/'),
        ('LPAREN',  r'\('),
        ('RPAREN',  r'\)'),
        ('SKIP',    r'[ \t]+'),
    ]
    tokens = []
    tok_regex = '|'.join(f'(?P<{name}>{pattern})' for name, pattern in token_spec)
    for mo in re.finditer(tok_regex, expression):
        kind = mo.lastgroup
        value = mo.group()
        if kind == 'NUMBER':
            value = float(value) if '.' in value else int(value)
            tokens.append(Token('NUMBER', value))
        elif kind == 'STAR':
            if tokens and tokens[-1].type == 'STAR':
                tokens[-1] = Token('POWER', '**')
            else:
                tokens.append(Token('STAR', value))
        elif kind in ('PLUS', 'MINUS', 'SLASH', 'LPAREN', 'RPAREN'):
            tokens.append(Token(kind, value))
        elif kind == 'SKIP':
            continue
    return tokens

The following are quick sanity checks of the tokenizer for known working expressions.

In [2]:
for token in tokenize('1 + 2 * 3'):
    print(token)

Token(NUMBER, 1)
Token(PLUS, +)
Token(NUMBER, 2)
Token(STAR, *)
Token(NUMBER, 3)


In [3]:
for token in tokenize("3 + 4 * 2 / (1 - 5) ** 2 ** 3"):
    print(token)

Token(NUMBER, 3)
Token(PLUS, +)
Token(NUMBER, 4)
Token(STAR, *)
Token(NUMBER, 2)
Token(SLASH, /)
Token(LPAREN, ()
Token(NUMBER, 1)
Token(MINUS, -)
Token(NUMBER, 5)
Token(RPAREN, ))
Token(POWER, **)
Token(NUMBER, 2)
Token(POWER, **)
Token(NUMBER, 3)


## Recursive Descent Parser

The parser is implemented using mutually recursive functions.
Each function corresponds to a grammar rule and returns the evaluated result of that rule.

Each function in the parser corresponds directly to a grammar rule.
The structure of the grammar has been carefully written to avoid any form of direct  recursion.
For example, the rule for `expr` does not recursively call itself as the first step on its right-hand side.
Instead, it first calls `term`, which in turn calls lower-precedence rules.
This ensures that every recursive path must consume at least one token before making another recursive call.
In recursive descent parsing, avoiding left recursion without first visiting an input symbol is essential to prevent infinite recursion during evaluation.

Parsing is performed by examining only a single token of lookahead.
This means that each function must decide what to do next based solely on the current token.
For instance, in the `expr` rule, once the parser has consumed an initial `term`, it checks whether the next token is a `+` or `-`.
If so, it consumes the operator and parses another `term`.
Otherwise, it finishes the expression.
This simple lookahead strategy is sufficient because the grammar has been designed so that each decision point is unambiguous with one token of lookahead.

Each grammar symbol is implemented as a function.
The body of the function follows the structure of the right-hand side of the grammar rule.
For example, the `expr` function first calls `term()` to process the first subexpression.
It then enters a loop to consume any remaining `+` or `-` tokens, repeatedly parsing additional `term` values.
Similarly, the `unary` function checks for optional unary `+` or `-` tokens and then calls itself recursively or defers to `primary` if no unary operator is present.

This systematic correspondence between grammar rules and functions makes the parser both readable and maintainable.
It also demonstrates the clarity and modularity that recursive descent parsing offers when working with well-structured grammars.

Note that the purpose of this example is intended for smaller languages.
When writing a parser for a larger language, there will be many more grammar rules and much more complexity (e.g. to handle variables, data types, control flow, and other more sophisticated language features).
Nevertheless, this minimal example shows the basic machinery that goes into writing compilers: tokenizing, parsing, and evaluating.

In [4]:
class ParserBasic:
    def __init__(self, tokens):
        self.tokens = tokens
        self.pos = 0

    def current(self):
        if self.pos < len(self.tokens):
            return self.tokens[self.pos]
        return Token('EOF', '')

    def eat(self, expected_type):
        token = self.current()
        if token.type == expected_type:
            self.pos += 1
            return token
        raise SyntaxError(f"Expected {expected_type}, got {token.type}")

    def parse(self):
        result = self.expr()
        if self.current().type != 'EOF':
            raise SyntaxError("Unexpected input after expression")
        return result

    def expr(self):
        value = self.term()
        while self.current().type in ('PLUS', 'MINUS'):
            op = self.eat(self.current().type).type
            right = self.term()
            value = value + right if op == 'PLUS' else value - right
        return value

    def term(self):
        value = self.factor()
        while self.current().type in ('STAR', 'SLASH'):
            op = self.eat(self.current().type).type
            right = self.factor()
            value = value * right if op == 'STAR' else value / right
        return value

    def factor(self):
        left = self.unary()
        if self.current().type == 'POWER':
            self.eat('POWER')
            right = self.factor()  # RECURSIVE CALL enables right-associativity
            return left ** right
        return left

    def unary(self):
        token = self.current()
        if token.type == 'PLUS':
            self.eat('PLUS')
            return +self.unary()
        elif token.type == 'MINUS':
            self.eat('MINUS')
            return -self.unary()
        else:
            return self.primary()

    def primary(self):
        token = self.current()
        if token.type == 'NUMBER':
            return self.eat('NUMBER').value
        elif token.type == 'LPAREN':
            self.eat('LPAREN')
            value = self.expr()
            self.eat('RPAREN')
            return value
        else:
            raise SyntaxError(f"Unexpected token: {token}")

## Unit Tests

The unit tests are organized into three distinct categories to promote clarity and comprehensive coverage.
The first group of tests, implemented in the method `test_valid_expressions()`, verifies that the parser correctly handles valid expressions by comparing the parser’s output to the expected numeric result.
These expressions range from simple literals like `42` to complex nested expressions involving multiple operators and parentheses.
Each test is independent and is run using Python’s `unittest` framework, which provides structure and detailed feedback in the case of failure.

The second group of tests is found in the method `test_invalid_expressions()`.
These inputs are deliberately malformed and are designed to confirm that the parser raises appropriate errors, specifically `SyntaxError`.
Examples include expressions with unmatched parentheses, incomplete operators, or sequences of tokens that do not form valid sub-expressions.
These tests ensure that the parser fails gracefully and predictably in the presence of incorrect input.

Finally, a third set of tests is implemented in the method `test_eval_equivalence()`.
This group performs a form of cross-validation by comparing the output of our parser with Python’s built-in `eval()` function on the same expressions.
This provides a strong assurance that our implementation respects standard operator precedence and associativity rules, since `eval()` reflects Python’s internal expression semantics.
These comparisons are especially useful for catching subtle errors that might otherwise go unnoticed in static test cases.


### Quick Sanity Check of Parser

Let's quickly test whether the parser is doing what it should be doing.

We will compare this to what Python would do, since we follow Python's arithmetic rules.

In [5]:
def eval_expr(expr):
    tokens = tokenize(expr)
    parser = ParserBasic(tokens)
    return parser.parse()

In [6]:
py_eval = eval("2 ** 3 ** 2")

In [7]:
our_eval = eval_expr("2 ** 3 ** 2")

In [8]:
print(f"Python: {py_eval}")
print(f"Our parser: {our_eval}")
print(f"Results match: {py_eval == our_eval}")

Python: 512
Our parser: 512
Results match: True


In [9]:
import unittest

class TestExpressionParser(unittest.TestCase):

    def eval_expr(self, expr):
        tokens = tokenize(expr)
        parser = ParserBasic(tokens)
        return parser.parse()

    def test_valid_expressions(self):
        cases = [
            ("42", 42),
            ("-5.5", -5.5),
            ("+(3)", 3),
            ("-(-7.25)", 7.25),
            ("2 + 3 * 4", 14),
            ("(2 + 3) * 4", 20),
            ("2 ** 3 ** 2", 512),
            ("-(3 + 4) * 2", -14),
            ("3 + 4 * 2 / (1 - 5) ** 2 ** 3", 3.0001220703125),
        ]
        for expr, expected in cases:
            with self.subTest(expr=expr):
                result = self.eval_expr(expr)
                self.assertAlmostEqual(result, expected, places=10)

    def test_invalid_expressions(self):
        cases = [
            "",
            "+",
            "()",
            "3 + * 4",
            "2 + (3 * 4",
            "2 **",
            "(2 + 3))",
            "3 3",
        ]
        for expr in cases:
            with self.subTest(expr=expr):
                with self.assertRaises(SyntaxError):
                    self.eval_expr(expr)

    def test_eval_equivalence(self):
        cases = [
            "42",
            "-5.5",
            "+(3)",
            "-(-7.25)",
            "2 + 3 * 4",
            "(2 + 3) * 4",
            "2 ** 3 ** 2",
            "-(3 + 4) * 2",
            "3 + 4 * 2 / (1 - 5) ** 2 ** 3",
        ]
        for expr in cases:
            with self.subTest(expr=expr):
                expected = eval(expr)
                result = self.eval_expr(expr)
                self.assertAlmostEqual(result, expected, places=10)

suite = unittest.TestLoader().loadTestsFromTestCase(TestExpressionParser)
unittest.TextTestRunner(verbosity=2).run(suite)



test_eval_equivalence (__main__.TestExpressionParser.test_eval_equivalence) ... ok
test_invalid_expressions (__main__.TestExpressionParser.test_invalid_expressions) ... ok
test_valid_expressions (__main__.TestExpressionParser.test_valid_expressions) ... ok

----------------------------------------------------------------------
Ran 3 tests in 0.010s

OK


<unittest.runner.TextTestResult run=3 errors=0 failures=0>

There are a few features of unittest that we are utilizing to make testing a bit more resilient.
These appear in the following few lines of code, excerpted from the above unit test.

```python
        for expr in cases:
            with self.subTest(expr=expr):
                with self.assertRaises(SyntaxError):
                    self.eval_expr(expr)
```

We make use of two `unittest` capabilities here.

- For testing multiple expressions within the same function, we make use of `self.subTest` from Python’s `unittest` framework. This allows us to loop over a set of test cases and run a separate subtest for each one. If a subtest fails, the failure is remembered by `unittest`, but the loop continues and the remaining subtests are still executed. At the end, the test function is marked as failed if any of the subtests fail. This approach makes it easier to detect and diagnose multiple issues in a single test run and provides clearer feedback, since the output identifies which specific input caused a failure.

- We also use `assertRaises` to test expressions that are expected to result in syntax errors with our previously defined `SyntaxError` exception. This method ensures that a particular exception is raised during a block of code. In our parser, malformed expressions typically raise a `SyntaxError` during parsing. Wrapping the parser call in `with self.assertRaises(SyntaxError):` confirms that the parser behaves correctly by detecting invalid input and failing in a controlled and expected way. This kind of test guards against silent failures or unintended behavior when the input violates the grammar.


## Recursive AST Construction

In most interpreters and compilers, it is standard practice to first transform the input source code into an *abstract syntax tree* (AST) rather than directly evaluating it during parsing. The AST provides a clean, structured representation of the input program that is independent of any specific processing step. Once constructed, the AST can be analyzed, transformed, or evaluated as needed. This separation of parsing from evaluation or code generation is essential in practical language compilers and interpreters.

For our arithmetic expression parser, we will extend it to build a simple AST that reflects the structure of the grammar. The tree will be composed of three kinds of nodes: numbers, unary operations, and binary operations. These correspond directly to the elements of our grammar and allow us to represent any valid expression in our language.


In [10]:
class NumberNode(object):
    def __init__(self, value):
        self.value = value
    def __repr__(self):
        return f"NumberNode({self.value})"

class UnaryOpNode(object):
    def __init__(self, operator, operand):
        self.operator = operator  # 'PLUS' or 'MINUS'
        self.operand = operand
    def __repr__(self):
        return f"UnaryOpNode({self.operator}, {self.operand})"


class BinaryOpNode(object):
    def __init__(self, left, operator, right):
        self.left = left
        self.operator = operator  # 'PLUS', 'MINUS', etc.
        self.right = right
    def __repr__(self):
        return f"BinaryOpNode({self.left}, {self.operator}, {self.right})"

To construct the AST during parsing, we revise each parser function to return an appropriate node instead of evaluating the expression. Each grammar rule corresponds to a function that builds and returns part of the tree. The rule for \verb|expr|, for example, constructs a \verb|BinaryOpNode| for each addition or subtraction operator it processes. The same applies to \verb|term|, \verb|factor|, and \verb|unary|, where we construct a \verb|BinaryOpNode| for the the \verb|term| and \verb|factor| rules and \verb|UnaryOpNode| for the \verb|unary|.


In [11]:
class ParserAST:
    def __init__(self, tokens):
        self.tokens = tokens
        self.pos = 0

    def current(self):
        if self.pos < len(self.tokens):
            return self.tokens[self.pos]
        return Token('EOF', '')

    def eat(self, expected_type):
        token = self.current()
        if token.type == expected_type:
            self.pos += 1
            return token
        raise SyntaxError(f"Expected {expected_type}, got {token.type}")

    def parse(self):
        result = self.expr()
        if self.current().type != 'EOF':
            raise SyntaxError("Unexpected input after expression")
        return result

    def expr(self):
        node = self.term()
        while self.current().type in ('PLUS', 'MINUS'):
            op = self.eat(self.current().type).type
            right = self.term()
            node = BinaryOpNode(node, op, right)
        return node

    def term(self):
        node = self.factor()
        while self.current().type in ('STAR', 'SLASH'):
            op = self.eat(self.current().type).type
            right = self.factor()
            node = BinaryOpNode(node, op, right)
        return node

    def factor(self):
        left = self.unary()
        if self.current().type == 'POWER':
            self.eat('POWER')
            right = self.factor()  # enables right-associativity
            return BinaryOpNode(left, 'POWER', right)
        return left

    def unary(self):
        token = self.current()
        if token.type == 'PLUS':
            self.eat('PLUS')
            return UnaryOpNode('PLUS', self.unary())
        elif token.type == 'MINUS':
            self.eat('MINUS')
            return UnaryOpNode('MINUS', self.unary())
        else:
            return self.primary()

    def primary(self):
        token = self.current()
        if token.type == 'NUMBER':
            return NumberNode(self.eat('NUMBER').value)
        elif token.type == 'LPAREN':
            self.eat('LPAREN')
            node = self.expr()
            self.eat('RPAREN')
            return node
        else:
            raise SyntaxError(f"Unexpected token: {token}")


## Recursive AST Evaluation


With the above AST, we now implement a separate function -- also recursive -- to evaluate it. This function walks the tree recursively and computes the numeric result of the expression. To do this cleanly, we use Python’s structural pattern matching (`match` / `case` introduced in Python 3.10. This feature allows us to match on both the type of node and its internal structure in a concise and readable way.


In [12]:
def evaluate(node):
    match node:
        case NumberNode(value=value):
            return value

        case UnaryOpNode(operator=operator, operand=operand):
            val = evaluate(operand)
            match operator:
                case 'PLUS':
                    return +val
                case 'MINUS':
                    return -val
                case _:
                    raise ValueError(f"Unknown unary operator: {operator}")

        case BinaryOpNode(left=left, operator=operator, right=right):
            lval = evaluate(left)
            rval = evaluate(right)
            match operator:
                case 'PLUS':
                    return lval + rval
                case 'MINUS':
                    return lval - rval
                case 'STAR':
                    return lval * rval
                case 'SLASH':
                    return lval / rval
                case 'POWER':
                    return lval ** rval
                case _:
                    raise ValueError(f"Unknown binary operator: {operator}")

        case int() | float():
            return node

        case _:
            raise TypeError(f"Unknown node type: {node}")

## Recursively Print the AST

In [13]:
def print_ast(node, indent=0):
    prefix = "  " * indent
    match node:
        case NumberNode(value=value):
            print(f"{prefix}Number({value})")
        case UnaryOpNode(operator=operator, operand=operand):
            print(f"{prefix}UnaryOpNode({operator})")
            print_ast(operand, indent + 1)
        case BinaryOpNode(left=left, operator=operator, right=right):
            print(f"{prefix}BinaryOpNode({operator})")
            print_ast(left, indent + 1)
            print_ast(right, indent + 1)
        case _:
            print(f"{prefix}Unknown node: {node}")

## Unit Tests for AST

Similar to the earlier unit tests for the parser with direct evaluation, this version constructs the AST and then evaluates the results identically to the earlier unit tests. The only difference is that we are building an AST and evaluating it separately.

### Quick Sanity Check of AST-based Parser

Similar to earlier, the following function shows how to put the tokenizer and parser together. We also show how to sanity check that the AST is valid for a simple expression.

In [14]:

def parse_and_eval_tree(expr):
    tokens = tokenize(expr)
    parser = ParserAST(tokens)
    tree = parser.parse()
    print_ast(tree)
    return evaluate(tree)

In [15]:
parse_and_eval_tree("2 + 3 ** 4 / 5 ** 6")

BinaryOpNode(PLUS)
  Number(2)
  BinaryOpNode(SLASH)
    BinaryOpNode(POWER)
      Number(3)
      Number(4)
    BinaryOpNode(POWER)
      Number(5)
      Number(6)


2.005184


This function is naturally recursive and mirrors the structure of the grammar. For each node, we match its type and extract its contents. The use of `match ... case` makes this both safer and more readable than traditional `if`/`elif` chains, and allows us to add new node types in the future with minimal changes. The explicit `raise` statements ensure that any deviation from expected structure or values results in a clear error.


In [16]:
import unittest

class TestExpressionParser(unittest.TestCase):

    def parse_and_evaluate(self, expr):
        tokens = tokenize(expr)
        parser = ParserAST(tokens)
        tree = parser.parse()
        return evaluate(tree)

    def test_valid_expressions(self):
        cases = [
            ("42", 42),
            ("-5.5", -5.5),
            ("+(3)", 3),
            ("-(-7.25)", 7.25),
            ("2 + 3 * 4", 14),
            ("(2 + 3) * 4", 20),
            ("2 ** 3 ** 2", 512),
            ("-(3 + 4) * 2", -14),
            ("3 + 4 * 2 / (1 - 5) ** 2 ** 3", 3.0001220703125),
        ]
        for expr, expected in cases:
            with self.subTest(expr=expr):
                result = self.parse_and_evaluate(expr)
                self.assertAlmostEqual(result, expected, places=10)

    def test_invalid_expressions(self):
        cases = [
            "",               # empty input
            "+",              # lone operator
            "()",             # empty parentheses
            "3 + * 4",        # bad operator usage
            "2 + (3 * 4",     # missing closing parenthesis
            "2 **",           # missing exponent
            "(2 + 3))",       # extra closing parenthesis
            "3 3",            # two numbers without operator
        ]
        for expr in cases:
            with self.subTest(expr=expr):
                with self.assertRaises(SyntaxError):
                    tokens = tokenize(expr)
                    parser = ParserAST(tokens)
                    tree = parser.parse()
                    _ = evaluate(tree)  # usually not reached

    def test_eval_equivalence(self):
        cases = [
            "42",
            "-5.5",
            "+(3)",
            "-(-7.25)",
            "2 + 3 * 4",
            "(2 + 3) * 4",
            "2 ** 3 ** 2",
            "-(3 + 4) * 2",
            "3 + 4 * 2 / (1 - 5) ** 2 ** 3",
        ]
        for expr in cases:
            with self.subTest(expr=expr):
                expected = eval(expr)
                result = self.parse_and_evaluate(expr)
                self.assertAlmostEqual(result, expected, places=10)

suite = unittest.TestLoader().loadTestsFromTestCase(TestExpressionParser)
unittest.TextTestRunner(verbosity=2).run(suite)



test_eval_equivalence (__main__.TestExpressionParser.test_eval_equivalence) ... ok
test_invalid_expressions (__main__.TestExpressionParser.test_invalid_expressions) ... ok
test_valid_expressions (__main__.TestExpressionParser.test_valid_expressions) ... ok

----------------------------------------------------------------------
Ran 3 tests in 0.012s

OK


<unittest.runner.TextTestResult run=3 errors=0 failures=0>

With the parser now returning an abstract syntax tree (AST) rather than immediately evaluating expressions, we revise our unit tests to reflect this separation of concerns. In the updated design, each test parses the input string to construct an AST and then evaluates the resulting tree using the standalone evaluator function. This differs from our earlier approach, where the parser directly returned a numeric value. The structure of the tests remains the same, with three categories: expressions that must evaluate correctly, expressions that are expected to raise parsing errors, and expressions whose results are compared with Python’s built-in \verb|eval()| function. However, each test now calls \verb|parse()| followed by \verb|evaluate()|, which provides a clearer separation between syntactic analysis and execution. This two-step approach better reflects how compilers and interpreters are structured and ensures that both tree construction and evaluation logic are tested independently and in combination.


In [17]:
import unittest

class TestExpressionParser(unittest.TestCase):

    def parse_and_evaluate(self, expr):
        tokens = tokenize(expr)
        parser = ParserAST(tokens)
        tree = parser.parse()
        return evaluate(tree)

    def test_valid_expressions(self):
        cases = [
            ("42", 42),
            ("-5.5", -5.5),
            ("+(3)", 3),
            ("-(-7.25)", 7.25),
            ("2 + 3 * 4", 14),
            ("(2 + 3) * 4", 20),
            ("2 ** 3 ** 2", 512),
            ("-(3 + 4) * 2", -14),
            ("3 + 4 * 2 / (1 - 5) ** 2 ** 3", 3.0001220703125),
        ]
        for expr, expected in cases:
            with self.subTest(expr=expr):
                result = self.parse_and_evaluate(expr)
                self.assertAlmostEqual(result, expected, places=10)

    def test_invalid_expressions(self):
        cases = [
            "",               # empty input
            "+",              # lone operator
            "()",             # empty parentheses
            "3 + * 4",        # bad operator usage
            "2 + (3 * 4",     # missing closing parenthesis
            "2 **",           # missing exponent
            "(2 + 3))",       # extra closing parenthesis
            "3 3",            # two numbers without operator
        ]
        for expr in cases:
            with self.subTest(expr=expr):
                with self.assertRaises(SyntaxError):
                    tokens = tokenize(expr)
                    parser = ParserAST(tokens)
                    tree = parser.parse()
                    _ = evaluate(tree)  # usually not reached

    def test_eval_equivalence(self):
        cases = [
            "42",
            "-5.5",
            "+(3)",
            "-(-7.25)",
            "2 + 3 * 4",
            "(2 + 3) * 4",
            "2 ** 3 ** 2",
            "-(3 + 4) * 2",
            "3 + 4 * 2 / (1 - 5) ** 2 ** 3",
        ]
        for expr in cases:
            with self.subTest(expr=expr):
                expected = eval(expr)
                result = self.parse_and_evaluate(expr)
                self.assertAlmostEqual(result, expected, places=10)

suite = unittest.TestLoader().loadTestsFromTestCase(TestExpressionParser)
unittest.TextTestRunner(verbosity=2).run(suite)


test_eval_equivalence (__main__.TestExpressionParser.test_eval_equivalence) ... ok
test_invalid_expressions (__main__.TestExpressionParser.test_invalid_expressions) ... ok
test_valid_expressions (__main__.TestExpressionParser.test_valid_expressions) ... ok

----------------------------------------------------------------------
Ran 3 tests in 0.010s

OK


<unittest.runner.TextTestResult run=3 errors=0 failures=0>

## Conclusion

This example introduced a recursive descent parser for arithmetic expressions. By defining a clean grammar and turning each rule into a          recursive function, we built a system that is easy to understand, extend, and debug. The resulting parser respects standard operator             precedence and associativity, and can evaluate expressions to match Python’s behavior.

  With this foundation, future work might include adding variables, assignment, or functions. But even in its current form, this parser captures   the essential ideas of recursive parsing and expression evaluation.